In [18]:
%load_ext autoreload
%autoreload 2

In [7]:
#  Copyright (c) Prior Labs GmbH 2025.

from __future__ import annotations

import logging
from copy import deepcopy
from typing import Any, Literal, Optional
from typing_extensions import Self

import pydantic
from pydantic import PositiveInt
from pydantic.dataclasses import dataclass

# from tabpfn.architectures.interface import ArchitectureConfig

logger = logging.getLogger(__name__)

FeaturePositionalEmbedding = Optional[
    Literal["normal_rand_vec", "uni_rand_vec", "learned", "subspace"]
]

@dataclass
class ArchitectureConfig:
    """Base configuration class that each architecture config should inherit from.

    Contains config keys common to all the architectures.
    """

    max_num_classes: int
    num_buckets: int
    """In regression models: the number of buckets in the output bar distribution.

    In classification models, does nothing.
    """

    def get_unused_config(self, unparsed_config: dict[str, Any]) -> dict[str, Any]:
        """Returns items in the given config that were not parsed by this config.

        This emulates Pydantic's extra="allow" and __pydantic_extra__ feature, which
        unfortunately isn't supported for dataclasses.
        """
        return _get_unused_items(full_config=unparsed_config, used_config=asdict(self))



@dataclass
class ModelConfig(ArchitectureConfig):
    """Configuration for the base architecture."""

    # ------ Actual variation across configs
    emsize: int = 192
    """The embedding dimension."""
    features_per_group: PositiveInt = 2
    """If > 1, the features will be grouped into groups of this size and the attention
    is across groups."""
    nhead: int = 6
    """Number of attention heads for both between-item and between-feature attention."""
    remove_duplicate_features: bool = False

    # --------

    # --- Constant across all configs and used
    dropout: float = 0.0
    encoder_use_bias: bool = False
    encoder_type: Literal["linear", "mlp"] = "linear"
    """Type of input encoder to use. Either "linear" for a simple linear layer or "mlp"
    for a multi-layer perceptron."""
    encoder_mlp_hidden_dim: int | None = 1024
    """Hidden dimension for MLP encoder. If None, defaults to emsize. Only used when
    encoder_type="mlp"."""
    encoder_mlp_num_layers: int = 2
    """Number of layers in the MLP encoder. Only used when encoder_type="mlp"."""
    feature_positional_embedding: FeaturePositionalEmbedding = "subspace"
    multiquery_item_attention: Literal[False] = False
    """When True, uses multiquery for attention between items."""
    nan_handling_enabled: Literal[True] = True
    nan_handling_y_encoder: Literal[True] = True
    nhid_factor: int = 4
    """Hidden dimension in the MLP layers is ninp * nhid_factor."""
    nlayers: int = 12
    """Number of layers in the encoder, each consisting of
    a multi-head attention and an MLP layer."""
    normalize_by_used_features: Literal[True] = True
    normalize_on_train_only: Literal[True] = True
    normalize_to_ranking: Literal[False] = False
    normalize_x: Literal[True] = True
    recompute_attn: bool = False
    """If True, enables activation checkpointing for each attention  layer **and each
    MLP layer** in the encoder. This saves memory. recompute_layer is a related flag
    which checkpoints the input to each PerFeatureEncoderLayer."""
    recompute_layer: bool = True
    """If True, enables activation checkpointing for each PerFeatureEncoderLayer in the
    encoder. This saves memory. recompute_attn is a related flag which checkpoints the
    attention and mlp layers individually. Note that the forward pass takes an argument
    `force_recompute_layer` which can be used to force recomputation of the layer."""
    remove_empty_features: Literal[True] = True
    remove_outliers: Literal[False] = False
    use_separate_decoder: Literal[False] = False
    """If True, the decoder will be separate from the encoder."""

    multiquery_item_attention_for_test_set: bool = True
    """If True, uses multiquery attention on the test set.
    For now, this must be False for bridge attention and True otherwise."""

    attention_init_gain: float = 1.0
    """The gain when initializing the attention parameters. If None, then 1.0 is
    used."""
    # --------

    dag_pos_enc_dim: int | None = None

    item_attention_type: Literal["full"] = "full"
    feature_attention_type: Literal["full"] = "full"
    seed: int = 0
    """The seed to use for the model. The default 0 is chosen to match
    the default random_state of 0 in the TabPFN estimator,
    which was used to set this seed before
    (though I'm not sure it makes a difference for a trained model).
    """

    num_thinking_rows: int = 0
    """If >0, then this number of "thinking rows" will be prepended to each dataset.
    See tabpfn.architectures.base.AddThinkingTokens for an explanation.
    """

    @classmethod
    def upgrade_config(cls, config: dict[str, Any]) -> dict[str, Any]:
        """Upgrade old configs to match the current config.

        This allows backwards compatibility with  checkpoints.
        Raises a ValueError if the config is not compatible with the current code.
        """
        # The dates are to help us remove upgrades when they get very old.
        config = deepcopy(config)

        # Config changed on unknown date
        try:
            del config["use_flash_attention"]
            logger.debug(
                "`use_flash_attention` was specified in the config. This will be "
                "ignored and the attention implementation selected automatically."
            )
        except KeyError:
            pass

        # Config changed on 2025-05-22
        # Some keys were previously allowed to be None, and replaced with a default
        # value when they were used. Now we keep the default value in the configs and
        # None isn't allowed, so replace None with the default value.
        if "attention_init_gain" in config and config["attention_init_gain"] is None:
            config["attention_init_gain"] = cls._get_default("attention_init_gain")

        # Config changed on 2025-06-03
        if "attention_type" in config:
            if "item_attention_type" in config or "feature_attention_type" in config:
                raise ValueError("Can't have both old and new attention types set")
            config["item_attention_type"] = config["attention_type"]
            config["feature_attention_type"] = config["attention_type"]
            del config["attention_type"]

        # Config changed on 2025-06-04
        if config.get("canonical_y_encoder", False) is not False:
            raise ValueError("Current version only supports canonical_y_encoder=False")
        if config.get("bias", False) is not False:
            raise ValueError("Current version only supports bias=False")

        # Config changed on 2025-07-09
        if config.pop("two_sets_of_queries", False):
            raise ValueError("`two_sets_of_queries` is no longer supported in config")

        return config

    @classmethod
    def _get_default(cls, field: str) -> Any:
        return cls.__dataclass_fields__[field].default

    @pydantic.model_validator(mode="after")
    def validate_consistent(self) -> Self:
        if self.emsize % self.nhead != 0:
            raise ValueError("emsize must be divisible by nhead")
        return self


In [30]:
#  Copyright (c) Prior Labs GmbH 2025.
"""Implements standard quadratic attention."""

from __future__ import annotations

import math
from functools import partial
from typing import TYPE_CHECKING
from typing_extensions import override

import torch
from torch.utils.checkpoint import checkpoint

# from tabpfn.architectures.base.attention import Attention
# from tabpfn.architectures.base.memory import support_save_peak_mem_factor

# if TYPE_CHECKING:
#     from tabpfn.architectures.base.config import ModelConfig

TORCH_VERSION = torch.__version__.split(".")

TORCH_2_ATTENTION_POSSIBLE = int(TORCH_VERSION[0]) >= 2
from __future__ import annotations

from abc import ABC, abstractmethod
from typing_extensions import override

import torch
from torch import nn


class Attention(ABC, nn.Module):
    """Base class for attention layers."""

    @override
    @abstractmethod
    def forward(
        self,
        x: torch.Tensor,
        x_kv: torch.Tensor | None = None,
        *,
        cache_kv: bool = False,
        use_cached_kv: bool = False,
        reuse_first_head_kv: bool = False,
        only_cache_first_head_kv: bool = False,
        # save_peak_mem_factor: int | None = None,
        # add_input: bool = False,
        # allow_inplace: bool = False,
    ) -> torch.Tensor:
        """Performs the attention layer.

        Args:
            x: Input sequence of embeddings with shape
                [batch... x query seq len x embedding dim].
                If `x_kv` is None, this is used to compute the queries, keys, and
                values.
                If `x_kv` is not None, this is used to compute the queries only.
            x_kv: If not None, an input sequence of embeddings with shape
                [batch... x kv seq len x embedding dim].
                It will be used to compute the keys and the values, with `x` used only
                to compute the queries. This is useful to avoid some sequence positions
                attending to others.
            cache_kv: If True, replaces the current key-value cache with the keys and
                values computed during this forward pass. Otherwise, the KV cache is
                left unchanged. If True, `use_cached_kv` must be False.
            use_cached_kv: If True, uses the keys and values cached during a previous
                forward pass when `cache_kv` was True. If True, `cache_kv` must be
                False.
            reuse_first_head_kv: If True, then uses the keys and values projected in the
                first head for all the heads.
            only_cache_first_head_kv: When True, only caches the keys and values of the
                first head.
            save_peak_mem_factor: Loops over the batch dimension rather than processing
                it in parallel, to reduce memory usage.
            add_input: If True, returns (x+the output of attention), i.e. enables a
                residual connection. If False, just returns the output of attention.
            allow_inplace: By setting this to True, the caller indicates that 'x' is not
                used after the call and its buffer can be reused for the output.
                The operation is not guaranteed to be inplace.

        Returns:
            Output hidden states of shape [batch x query seq len x embedding dim]
        """
        ...


def _gqa_is_supported() -> bool:
    """Check if PyTorch's scaled_dot_product_attention supports enable_gqa parameter.

    This checks whether torch.nn.functional.scaled_dot_product_attention has a
    kwarg enable_gqa and if we have sufficient NVIDIA compute capability.
    PyTorch 2.5+ includes enable_gqa support.
    """
    if not TORCH_2_ATTENTION_POSSIBLE or not torch.cuda.is_available():
        return False

    # Check if PyTorch version is 2.5 or higher for enable_gqa support
    torch_major, torch_minor = int(TORCH_VERSION[0]), int(TORCH_VERSION[1])
    has_enable_gqa = torch_major > 2 or (torch_major == 2 and torch_minor >= 5)

    if not has_enable_gqa:
        return False

    # Check compute capability only if CUDA is available
    # We need compute capability >= 8.0 for efficient GQA
    device = torch.cuda.current_device()
    nvidia_compute_capability = torch.cuda.get_device_capability(device)
    return nvidia_compute_capability[0] >= 8


# Cache the GQA support check at module level
USE_TORCH_2_GQA = _gqa_is_supported()


class PFNMultiHeadAttention(Attention):
    _input_size: int
    _output_size: int
    _nhead: int
    _nhead_kv: int
    _d_k: int
    _d_v: int
    _share_kv_across_n_heads: int
    dropout_p: float | None
    softmax_scale: float | None
    _k_cache: torch.Tensor | None
    _v_cache: torch.Tensor | None
    _kv_cache: torch.Tensor | None
    _w_q: torch.nn.Parameter | None
    _w_k: torch.nn.Parameter | None
    _w_v: torch.nn.Parameter | None
    _w_kv: torch.nn.Parameter | None
    _w_qkv: torch.nn.Parameter | None
    _w_out: torch.nn.Parameter

    @property
    def w_q(self) -> torch.nn.Parameter | None:
        return self._w_q

    @property
    def w_k(self) -> torch.nn.Parameter | None:
        return self._w_k

    @property
    def w_v(self) -> torch.nn.Parameter | None:
        return self._w_v

    @property
    def w_qkv(self) -> torch.nn.Parameter | None:
        return self._w_qkv

    @property
    def w_kv(self) -> torch.nn.Parameter | None:
        return self._w_kv

    @property
    def w_out(self) -> torch.nn.Parameter:
        return self._w_out

    @property
    def has_cached_kv(self) -> bool:
        assert (self._k_cache is None) == (self._v_cache is None)
        assert self._kv_cache is None or (
            self._k_cache is None and self._v_cache is None
        )
        return (
            self._k_cache is not None and self._v_cache is not None
        ) or self._kv_cache is not None

    def empty_kv_cache(self) -> None:
        self._k_cache = None
        self._v_cache = None
        self._kv_cache = None

    def set_parameters(
        self,
        w_out: torch.nn.Parameter,
        w_q: torch.nn.Parameter | None = None,
        w_k: torch.nn.Parameter | None = None,
        w_v: torch.nn.Parameter | None = None,
        w_kv: torch.nn.Parameter | None = None,
        w_qkv: torch.nn.Parameter | None = None,
        precomputed_k: torch.Tensor | None = None,
        precomputed_v: torch.Tensor | None = None,
        precomputed_kv: torch.Tensor | None = None,
    ) -> None:
        assert (precomputed_k is None) == (precomputed_v is None)
        assert (precomputed_kv is None) or (precomputed_k is None)
        assert (precomputed_kv is None and precomputed_k is None) != (
            w_qkv is None and w_kv is None and w_k is None and w_v is None
        )
        assert (w_qkv is None) != (w_q is None)
        assert (w_qkv is None) or (w_kv is None and w_k is None and w_v is None)
        assert w_kv is None or (w_k is None and w_v is None)
        assert (w_k is None) == (w_v is None)

        def assert_tensor_shape(
            tensor: torch.Tensor | None,
            expected_shape: list[int | None],
        ) -> None:
            if tensor is None:
                return
            actual_shape = tensor.size()
            err = f"Tensor {actual_shape=} does not match {expected_shape=}."
            assert len(actual_shape) == len(expected_shape), err
            for actual_dim, expected_dim in zip(actual_shape, expected_shape):
                if expected_dim is not None:
                    assert actual_dim == expected_dim, err

        assert_tensor_shape(precomputed_k, [None, None, self._nhead_kv, self._d_k])
        assert_tensor_shape(precomputed_v, [None, None, self._nhead_kv, self._d_v])
        assert_tensor_shape(precomputed_kv, [None, None, 2, self._nhead_kv, self._d_k])
        assert_tensor_shape(w_q, [1, self._nhead, self._d_k, self._input_size])
        assert_tensor_shape(w_k, [self._nhead_kv, self._d_k, self._input_size])
        assert_tensor_shape(w_v, [self._nhead_kv, self._d_v, self._input_size])
        assert_tensor_shape(w_kv, [2, self._nhead_kv, self._d_k, self._input_size])
        assert_tensor_shape(w_qkv, [3, self._nhead, self._d_k, self._input_size])
        assert_tensor_shape(w_out, [self._nhead, self._d_v, self._output_size])

        self.register_parameter("_w_out", w_out)
        self.register_parameter("_w_q", w_q)
        self.register_parameter("_w_k", w_k)
        self.register_parameter("_w_v", w_v)
        self.register_parameter("_w_kv", w_kv)
        self.register_parameter("_w_qkv", w_qkv)

        self.register_buffer("_k_cache", precomputed_k)
        self.register_buffer("_v_cache", precomputed_v)
        self.register_buffer("_kv_cache", precomputed_kv)

    def newly_initialized_input_weight(
        self,
        dims: list[int],
        nhead: int,
        device: torch.device | None,
        dtype: torch.dtype | None,
    ) -> torch.nn.Parameter:
        assert 3 <= len(dims) <= 4  # ([stack,] nhead_, d, input_size)
        w = torch.nn.Parameter(torch.empty(*dims, device=device, dtype=dtype))
        d, input_size = dims[-2:]
        std = math.sqrt(2.0 / float(nhead * d + input_size)) * self.init_gain
        a = math.sqrt(3.0) * std
        torch.nn.init.uniform_(w, -a, a)
        return w

    def __init__(  # noqa: PLR0913
        self,
        *,
        d_k: int,
        d_v: int,
        device: torch.device | None,
        dtype: torch.dtype | None,
        config: ModelConfig,
        share_kv_across_n_heads: int = 1,
        dropout_p: float | None = None,
        softmax_scale: float | None = None,
        initialize_output_to_zero: bool = False,
        precomputed_k: torch.Tensor | None = None,
        precomputed_v: torch.Tensor | None = None,
        precomputed_kv: torch.Tensor | None = None,
    ):
        super().__init__()
        assert config.nhead % share_kv_across_n_heads == 0
        self._input_size = config.emsize
        self._output_size = config.emsize
        self._d_k = d_k
        self._d_v = d_v
        self._nhead = config.nhead
        self._nhead_kv = config.nhead // share_kv_across_n_heads
        self._device = device
        self._dtype = dtype
        self.dropout_p = dropout_p
        self.softmax_scale = softmax_scale
        self.init_gain = config.attention_init_gain

        w_out = torch.nn.Parameter(
            torch.empty(
                config.nhead, d_v, self._output_size, device=device, dtype=dtype
            ),
        )
        if initialize_output_to_zero:
            torch.nn.init.zeros_(w_out)
        else:
            torch.nn.init.xavier_uniform_(w_out)

        assert precomputed_k is None == precomputed_v is None
        has_precomputed_kv = precomputed_kv is not None or precomputed_k is not None
        w_q = None
        w_k = None
        w_v = None
        w_kv = None
        w_qkv = None
        if d_k == d_v and self._nhead == self._nhead_kv and not has_precomputed_kv:
            w_qkv = self.newly_initialized_input_weight(
                [3, self._nhead, self._d_k, self._input_size],
                nhead=self._nhead,
                device=device,
                dtype=dtype,
            )
        else:
            w_q = self.newly_initialized_input_weight(
                [1, self._nhead, self._d_k, self._input_size],
                nhead=self._nhead,
                device=device,
                dtype=dtype,
            )
            if not has_precomputed_kv:
                if d_k == d_v:
                    w_kv = self.newly_initialized_input_weight(
                        [2, self._nhead_kv, self._d_k, self._input_size],
                        nhead=self._nhead,
                        device=device,
                        dtype=dtype,
                    )
                else:
                    w_k = self.newly_initialized_input_weight(
                        [self._nhead_kv, self._d_k, self._input_size],
                        nhead=self._nhead,
                        device=device,
                        dtype=dtype,
                    )
                    w_v = self.newly_initialized_input_weight(
                        [self._nhead_kv, self._d_v, self._input_size],
                        nhead=self._nhead,
                        device=device,
                        dtype=dtype,
                    )
        self.set_parameters(
            w_out,
            w_q,
            w_k,
            w_v,
            w_kv,
            w_qkv,
            precomputed_k,
            precomputed_v,
            precomputed_kv,
        )
        if config.recompute_attn:
            self.forward = partial(checkpoint, self.forward, use_reentrant=False)  # type: ignore

    @override
    def forward(
        self,
        x: torch.Tensor,
        x_kv: torch.Tensor | None = None,
        *,
        cache_kv: bool = False,
        add_input: bool = False,
        # Indicates that 'x' is not used after the call and its buffer can be reused
        # for the output. The operation is not guaranteed to be inplace.
        allow_inplace: bool = False,
        # This requires 'add_input' and 'allow_inplace'. See the documentation of
        # the decorator 'support_save_peak_mem_factor' for details.
        # save_peak_mem_factor: int | None = None,
        reuse_first_head_kv: bool = False,
        only_cache_first_head_kv: bool = False,
        use_cached_kv: bool = False,
    ) -> torch.Tensor:
        """X is the current hidden and has a shape of [batch, ..., seq_len, input_size].
        If keys and values are present in the cache and 'freeze_kv' is not set, they
        are obtained from there and 'x_kv' has to be None.
        Else, if 'x_kv' is not None, keys and values are obtained by applying the
        respective linear transformations to 'x_kv'.
        Else, keys and values are attained by applying the respective linear
        transformations to 'x' (self attention).
        """
        assert not (cache_kv and use_cached_kv), (
            "Cannot cache and use cached keys and values at the same time."
        )

        assert not x.requires_grad or (not self.has_cached_kv and not cache_kv), (
            "Saving keys and values is only supported during inference."
        )
        x, x_kv, x_shape_after_transpose = self._rearrange_inputs_to_flat_batch(x, x_kv)

        nhead_kv = 1 if reuse_first_head_kv else self._nhead_kv

        if cache_kv:
            # Reset cache first so memory is freed before new cache is allocated.
            self._k_cache = self._v_cache = self._kv_cache = None

            if x_kv is not None:
                batch_size, seqlen_kv = x_kv.shape[:2]
            else:
                batch_size, seqlen_kv = x.shape[:2]

            # TODO: handling of device and dtype.
            if self._w_kv is not None or self._w_qkv is not None:
                self._kv_cache = torch.empty(
                    batch_size,
                    seqlen_kv,
                    2,
                    1 if only_cache_first_head_kv else nhead_kv,
                    self._d_k,
                    device=x.device,
                    dtype=x.dtype,
                )
            else:
                self._k_cache = torch.empty(
                    batch_size,
                    seqlen_kv,
                    nhead_kv,
                    self._d_k,
                    device=x.device,
                    dtype=x.dtype,
                )
                self._v_cache = torch.empty(
                    batch_size,
                    seqlen_kv,
                    nhead_kv,
                    self._d_v,
                    device=x.device,
                    dtype=x.dtype,
                )

        output: torch.Tensor = self._compute(
            x,
            x_kv,
            self._k_cache,
            self._v_cache,
            self._kv_cache,
            cache_kv=cache_kv,
            use_cached_kv=use_cached_kv,
            # add_input=add_input,
            # allow_inplace=allow_inplace,
            # save_peak_mem_factor=save_peak_mem_factor,
            reuse_first_head_kv=reuse_first_head_kv,
        )
        return output.reshape(x_shape_after_transpose[:-1] + output.shape[-1:])

    def compute_qkv(  # noqa: PLR0912, C901
        self,
        x: torch.Tensor,
        x_kv: torch.Tensor | None,
        k_cache: torch.Tensor | None,
        v_cache: torch.Tensor | None,
        kv_cache: torch.Tensor | None,
        *,
        cache_kv: bool,
        use_cached_kv: bool,
        reuse_first_head_kv: bool,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
        torch.Tensor | None,
        torch.Tensor | None,
    ]:
        assert not (cache_kv and use_cached_kv), (
            "You cannot both cache new KV and use the cached KV at once."
        )
        if reuse_first_head_kv:
            assert x is not x_kv, (
                "x and x_kv must be different tensors. That means reuse_first_head_kv"
                "is not compatible with self attention only cross attention."
            )
        if x_kv is None:
            x_kv = x

        k = v = kv = None
        if use_cached_kv:
            assert self.has_cached_kv, (
                "You try to use cached keys and values but the cache is empty."
            )
            k = k_cache
            v = v_cache
            kv = kv_cache

        assert (k is None) == (v is None)

        if self._w_qkv is None:
            w_q, w_kv = self._w_q[0], self._w_kv
        else:
            w_q, w_kv = self._w_qkv[0], self._w_qkv[1:]

        if (
            self._w_qkv is not None
            and x is x_kv
            and kv is None
            and k is None
            and v is None
        ):
            # A faster version of
            # qkv = torch.einsum("... s, j h d s -> ... j h d", x, self._w_qkv)
            batch_shape = x.shape[:-1]  # [..., seq_len]
            j, nhead, d_k, input_size = self._w_qkv.shape

            # [j, nhead, d_k, input_size] -> [j * nhead * d_k, input_size]
            w_flat = self._w_qkv.reshape(-1, input_size)

            qkv_flat = torch.matmul(x, w_flat.T)

            # Reshape back to desired format: [..., seq_len, j, nhead, d_k]
            qkv = qkv_flat.reshape(*batch_shape, j, nhead, d_k)
            q = None
        else:
            qkv = None
            q = torch.einsum("... s, h d s -> ... h d", x, w_q)

        if kv is None and k is None and v is None and qkv is None:
            if w_kv is not None:
                if reuse_first_head_kv:
                    orig_num_heads = w_kv.shape[1]
                    w_kv = w_kv[:, :1]
                kv = torch.einsum("... s, j h d s -> ... j h d", x_kv, w_kv)
                if reuse_first_head_kv:
                    expand_shape = [-1 for _ in kv.shape]
                    expand_shape[-2] = orig_num_heads
                    kv = kv.expand(*expand_shape)
            else:
                w_k = self._w_k
                w_v = self._w_v
                if reuse_first_head_kv:
                    orig_num_heads = w_k.shape[0]
                    w_k = w_k[:1]
                    w_v = w_v[:1]
                k = torch.einsum("... s, h d s -> ... h d", x_kv, w_k)
                v = torch.einsum("... s, h d s -> ... h d", x_kv, w_v)
                if reuse_first_head_kv:
                    expand_shape = [-1 for _ in k.shape]
                    expand_shape[-2] = orig_num_heads
                    k = k.expand(*expand_shape)
                    v = v.expand(*expand_shape)

        if cache_kv:
            if k_cache is not None:
                k_cache[:] = k
            if v_cache is not None:
                v_cache[:] = v
            if kv_cache is not None:
                if kv_cache.shape[-2] == 1:
                    # we are in the case where only the first head kv is cached
                    # that is the case when we only neeed that for inference
                    kv_cache[:] = kv[..., :1, :]
                else:
                    kv_cache[:] = kv

        return q, k, v, kv, qkv

    # @support_save_peak_mem_factor  # type: ignore
    def _compute(
        self,
        x: torch.Tensor,
        x_kv: torch.Tensor | None,
        k_cache: torch.Tensor | None,
        v_cache: torch.Tensor | None,
        kv_cache: torch.Tensor | None,
        *,
        cache_kv: bool,
        use_cached_kv: bool,
        reuse_first_head_kv: bool,
    ) -> torch.Tensor:
        """Attention computation.
        Called by 'forward', potentially on shards, once shapes have been normalized.
        """
        q, k, v, kv, qkv = self.compute_qkv(
            x,
            x_kv,
            k_cache,
            v_cache,
            kv_cache,
            cache_kv=cache_kv,
            use_cached_kv=use_cached_kv,
            reuse_first_head_kv=reuse_first_head_kv,
        )
        attention_head_outputs = MultiHeadAttention.compute_attention_heads(
            q,
            k,
            v,
            kv,
            qkv,
            self.dropout_p,
            self.softmax_scale,
        )
        return torch.einsum(
            "... h d, h d s -> ... s",
            attention_head_outputs,
            self._w_out,
        )

    def _rearrange_inputs_to_flat_batch(
        self,
        x: torch.Tensor,
        x_kv: torch.Tensor | None,
    ) -> tuple[torch.Tensor, torch.Tensor | None, torch.Size]:
        # TODO: This presumably creates potential memory overhead not captured
        # by save_peak_mem_factor.
        x_shape_after_transpose = x.shape
        if x_kv is not None:
            assert x.shape[:-2] == x_kv.shape[:-2]
        x = x.reshape(-1, *x.shape[-2:])
        if x_kv is not None:
            x_kv = x_kv.reshape(-1, *x_kv.shape[-2:])
        return x, x_kv, x_shape_after_transpose

    @staticmethod
    def broadcast_kv_across_heads(
        kv: torch.Tensor,
        share_kv_across_n_heads: int,
    ) -> torch.Tensor:
        if share_kv_across_n_heads == 1:
            return kv

        nhead, d = kv.shape[-2:]
        kv = kv[..., None, :].expand(
            *([-1] * (kv.dim() - 1)),
            share_kv_across_n_heads,
            -1,
        )
        return kv.reshape(*kv.shape[:-3], nhead * share_kv_across_n_heads, d)

    @staticmethod
    def scaled_dot_product_attention_chunked(
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        dropout_p: float | None = None,
        max_batch_size: int = 65_000,
        **extra_inputs,
    ) -> torch.Tensor:
        """Scaled dot product attention with automatic chunking to handle
        batch size limitations when batch size is larger than 65_535.
        This is a workaround for the issue: https://github.com/pytorch/pytorch/issues/142228.

        Args:
            q: Query tensor
            k: Key tensor
            v: Value tensor
            dropout_p: Dropout probability
            max_batch_size: Maximum batch size for CUDA kernels (default 65_000)
            extra_inputs: Additional arguments for scaled_dot_product_attention

        Returns:
            Attention output with same shape as input q
        """
        batch_size = q.shape[0]
        output_chunks = []

        for start_idx in range(0, batch_size, max_batch_size):
            end_idx = min(start_idx + max_batch_size, batch_size)

            q_chunk = q[start_idx:end_idx]
            k_chunk = k[start_idx:end_idx]
            v_chunk = v[start_idx:end_idx]

            chunk_output = torch.nn.functional.scaled_dot_product_attention(
                q_chunk,
                k_chunk,
                v_chunk,
                dropout_p=dropout_p,
                **extra_inputs,
            )

            output_chunks.append(chunk_output)

        # Concatenate results along batch dimension
        return torch.cat(output_chunks, dim=0)

    @staticmethod
    def compute_attention_heads(
        q: torch.Tensor | None,
        k: torch.Tensor | None,
        v: torch.Tensor | None,
        kv: torch.Tensor | None,
        qkv: torch.Tensor | None,
        dropout_p: float | None = None,
        softmax_scale: float | None = None,
        # text enhanced attention weight
        attn_weight_external: torch.Tensor | None = None,
        # weight between numerical attention weight & text attention weight
        external_gate: float | None = None,                
    ) -> torch.Tensor:
        assert (k is None) == (v is None)
        assert sum([qkv is None, kv is None, k is None and v is None]) == 2
        assert (qkv is None) != (q is None)

        if qkv is not None:
            q, k, v = qkv.unbind(dim=-3)
        elif kv is not None:
            k, v = kv.unbind(dim=-3)

        assert q is not None
        assert k is not None
        assert v is not None

        # checks if both attn_weight_external & external_gate are present or not
        assert (attn_weight_external is None) == (external_gate is None)

        batch_size, seqlen_q, nhead, d_k = q.shape
        _, _seqlen_kv, nhead_kv, d_v = v.shape
        share_kv_across_n_heads = nhead // nhead_kv
        if dropout_p is None:
            dropout_p = 0.0  # TODO: necessary?

        # use external attn or not
        use_external = attn_weight_external is not None and external_gate is not None

        # if there is no attn_weight passed to the function, use original falsh attn
        if TORCH_2_ATTENTION_POSSIBLE and not use_external:
            extra_inputs = {}
            if softmax_scale is not None:
                extra_inputs["scale"] = (
                    softmax_scale  # defaults to 1/sqrt(d_k) if None or not provided
                )

            # Check if we should use PyTorch 2.0's GQA support
            if USE_TORCH_2_GQA:
                extra_inputs["enable_gqa"] = True
            else:
                k = MultiHeadAttention.broadcast_kv_across_heads(
                    k,
                    share_kv_across_n_heads,
                )
                v = MultiHeadAttention.broadcast_kv_across_heads(
                    v,
                    share_kv_across_n_heads,
                )

            attention_head_outputs = (
                MultiHeadAttention.scaled_dot_product_attention_chunked(
                    q.transpose(1, 2),
                    k.transpose(1, 2),
                    v.transpose(1, 2),
                    dropout_p=dropout_p,
                    **extra_inputs,
                )
            )
            attention_head_outputs = attention_head_outputs.transpose(1, 2)
        # use external attention
        # the attention is calculated manually by broadcasting K, V to multiple heads.
        # the logits are available, we could use the logits for other fusion methods
        else:
            k = MultiHeadAttention.broadcast_kv_across_heads(k, share_kv_across_n_heads)
            v = MultiHeadAttention.broadcast_kv_across_heads(v, share_kv_across_n_heads)
            logits = torch.einsum("b q h d, b k h d -> b q k h", q, k)
            # apply d_k^1/2 to the logit
            logits *= (
                torch.sqrt(torch.tensor(1.0 / d_k)).to(k.device)
                if softmax_scale is None
                else softmax_scale
            )
            # apply softmax
            ps = torch.softmax(logits, dim=2)
            ps = torch.dropout(ps, dropout_p, train=True)
            
            
            if use_external:
                # attn_external = attn_external.permute(0, 2, 3, 1)
                text_enhanced_attn_weight = attn_weight_external.unsqueeze(1).repeat(1, nhead, 1, 1) # type: ignore
                # convex combination
                ps = external_gate * ps + (1.0 - external_gate) * text_enhanced_attn_weight # pyright: ignore[reportOptionalOperand, reportOperatorIssue]
            
            attention_head_outputs = torch.einsum("b q k h, b k h d -> b q h d", ps, v)

        
        return attention_head_outputs.reshape(
            batch_size,
            seqlen_q,
            nhead,
            d_v,
        )

    @staticmethod
    def convert_torch_nn_multihead_attention_state_dict(
        state_dict: dict,
        nhead: int,
        *,
        disable_stacked_w_qkv: bool = False,
    ) -> dict:
        in_proj_weight = state_dict["in_proj_weight"]
        out_proj_weight = state_dict["out_proj.weight"]

        embed_dim = in_proj_weight.shape[1]
        assert embed_dim % nhead == 0
        assert in_proj_weight.shape[0] == 3 * embed_dim
        assert out_proj_weight.shape == (embed_dim, embed_dim)
        in_proj_weight = in_proj_weight.reshape(3, nhead, -1, embed_dim)

        state_dict = {}
        if disable_stacked_w_qkv:
            state_dict["_w_q"], state_dict["_w_kv"] = torch.split(
                in_proj_weight,
                [1, 2],
            )
            state_dict["_w_q"] = state_dict["_w_q"].squeeze(0)
        else:
            state_dict["_w_qkv"] = in_proj_weight
        state_dict["_w_out"] = out_proj_weight.T.reshape(nhead, -1, embed_dim)
        return state_dict


In [31]:
import torch
import torch.nn as nn

class PFNMultiHeadAttentionWrapper(nn.Module):
    """
    Wrapper around PFNMultiHeadAttention to mimic torch.nn.MultiheadAttention.
    Uses compute_attention_heads internally.
    """

    def __init__(self, embed_dim: int, num_heads: int, config, device=None, dtype=None):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim

        # Initialize PFNMultiHeadAttention
        self.mha = PFNMultiHeadAttention(
            d_k=self.head_dim,
            d_v=self.head_dim,
            device=device,
            dtype=dtype,
            config=config,
        )

    def forward(self, query, key, value, need_weights=False):
        # Expect inputs in (batch, seq, embed)
        B, L, C = query.shape

        # Use PFN forward to get output
        output = self.mha(query, x_kv=key)

        attn_weights = None
        if need_weights:
            # Project Q, K, V using PFN's weights
            q_proj = torch.einsum("b s c, h d c -> b s h d", query, self.mha.w_q) \
                if self.mha.w_q is not None else None
            if self.mha.w_kv is not None:
                kv_proj = torch.einsum("b s c, j h d c -> b s j h d", key, self.mha.w_kv)
                k_proj, v_proj = kv_proj.unbind(dim=2)
            else:
                k_proj = torch.einsum("b s c, h d c -> b s h d", key, self.mha.w_k)
                v_proj = torch.einsum("b s c, h d c -> b s h d", key, self.mha.w_v)

            # Compute attention heads using PFN function
            attn_out = PFNMultiHeadAttention.compute_attention_heads(
                q_proj, k_proj, v_proj, kv=None, qkv=None
            )

            # Compute attention weights
            logits = torch.einsum("b s h d, b t h d -> b s t h", q_proj, k_proj)
            logits = logits / (self.head_dim ** 0.5)
            attn_weights = torch.softmax(logits, dim=2)
            attn_weights = attn_weights.permute(0, 3, 1, 2)  # (B, H, Lq, Lk)

        return output, attn_weights

In [32]:
import torch
import torch.nn as nn


class PFN_MHA_Wrapper(nn.Module):
    """
    Wrapper that makes PFNMultiHeadAttention behave like torch.nn.MultiheadAttention
    but uses PFNMultiHeadAttention.compute_attention_heads() internally.
    """

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        config,
        *,
        batch_first: bool = True,
        device=None,
        dtype=None,
    ):
        super().__init__()
        self.batch_first = batch_first
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim

        # ---- Create the PFN MHA module ------------------------------------
        self.mha = PFNMultiHeadAttention(
            d_k=self.head_dim,
            d_v=self.head_dim,
            device=device,
            dtype=dtype,
            config=config,
        )

    # ----------------------------------------------------------------------
    # FORWARD
    # ----------------------------------------------------------------------
    def forward(
        self,
        query,
        key,
        value,
        key_padding_mask=None,
        need_weights=False,
        attn_mask=None,
    ):

        # Convert query/key/value: (L,B,C) -> (B,L,C)
        if not self.batch_first:
            query = query.transpose(0, 1)
            key = key.transpose(0, 1)
            value = value.transpose(0, 1)

        # ---- 1) Use PFN forward for attention output ----------------------
        output = self.mha(query, x_kv=key)

        # ---- 2) If needed, compute weights using PFN.compute_attention_heads
        if need_weights:
            # PFN pre-projection into Q/K/V
            # Shapes: (B, L, H, D_head)
            q_proj = self.mha.linear_q(query) \
                .view(query.shape[0], query.shape[1], self.num_heads, self.head_dim)
            k_proj = self.mha.linear_k(key) \
                .view(key.shape[0], key.shape[1], self.num_heads, self.head_dim)
            v_proj = self.mha.linear_v(key) \
                .view(key.shape[0], key.shape[1], self.num_heads, self.head_dim)

            # PFN’s function returns: (B, L, H, D_head)
            attn_out = PFNMultiHeadAttention.compute_attention_heads(
                q=q_proj,
                k=k_proj,
                v=v_proj,
                kv=None,
                qkv=None,
                dropout_p=None,
                softmax_scale=None,
            )

            # To extract weights, we recompute logits → softmax using PFN math
            # LOGITS: (B, Lq, Lk, H)
            logits = torch.einsum(
                "blhd, bkhd -> blkh",
                q_proj,
                k_proj,
            ) / (self.head_dim**0.5)

            attn_weights = logits.softmax(dim=2)  # (B, Lq, Lk, H)

            # Convert weights to standard form: (B, H, Lq, Lk)
            attn_weights = attn_weights.permute(0, 3, 1, 2)

        else:
            attn_weights = None

        # Convert output back to (L,B,C)
        if not self.batch_first:
            output = output.transpose(0, 1)

        return output, attn_weights

In [33]:
# initialize the model config
config = ModelConfig(**{'max_num_classes': 10, 'num_buckets': 100,})
batch_size = 2
seq_len = 5
embed_dim = 16
num_heads = 4


mha = PFN_MHA_Wrapper(embed_dim=embed_dim, num_heads=num_heads, config=config, batch_first=True)

# Random input
x = torch.randn(batch_size, seq_len, embed_dim)

# Key padding mask (masking last position for batch 0)
key_padding_mask = torch.tensor([[0, 0, 0, 0, 1],
                                    [0, 0, 0, 0, 0]], dtype=torch.bool)

# Attention mask (prevent attention to future tokens)
attn_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

# Forward pass
out, attn_weights = mha(
    query=x,
    key=x,
    value=x,
    key_padding_mask=key_padding_mask,
    attn_mask=attn_mask,
    need_weights=True,
)

print("Output shape:", out.shape)  # Expect [batch, seq_len, embed_dim]
if attn_weights is not None:
    print("Attention weights shape:", attn_weights.shape)  # Expect [batch, num_heads, seq_len, seq_len]

# Simple check for residual connection (optional)
assert out.shape == (batch_size, seq_len, embed_dim), "Output shape mismatch"
print("PFNMHAWrapper test passed!")



RuntimeError: einsum(): subscript s has size 192 for operand 1 which does not broadcast with previously seen size 16

In [4]:
import torch
from tfmplayground.attn_v2_wrapper import PFNMultiheadAttentionV2Wrapper
from tfmplayground.attn_v2 import PFNAttentionConfig

torch.manual_seed(0)

batch_size, seq_len, embed_dim, num_heads = 2, 6, 16, 4
config = PFNAttentionConfig(emsize=embed_dim, nhead=num_heads)

mha_pfn_v2 = PFNMultiheadAttentionV2Wrapper(
    embed_dim=embed_dim,
    num_heads=num_heads,
    batch_first=True,
    bias=False,
    config=config,
)

query = torch.randn(batch_size, seq_len, embed_dim)
# PFN attention expects key and value to be the same tensor
key = value = torch.randn(batch_size, seq_len, embed_dim)

out, attn = mha_pfn_v2(query, key, value, need_weights=True)
print("out shape:", out.shape)    # (B, L, C)
print("attn shape:", attn.shape)  # (B, H, Lq, Lk) when need_weights=True
print("out sum:", out.sum().item())

out shape: torch.Size([2, 6, 16])
attn shape: torch.Size([2, 6, 6])
out sum: 6.88444185256958


In [2]:
import torch
from torch import nn

from tfmplayground.attn_v2 import PFNAttentionConfig, MultiHeadAttention as PFNMultiHeadAttentionV2
from tfmplayground.attn_v2_wrapper import PFNMultiheadAttentionV2Wrapper

torch.manual_seed(0)

# Shapes
batch_size, seq_len, embed_dim, num_heads = 2, 6, 16, 4
cfg = PFNAttentionConfig(emsize=embed_dim, nhead=num_heads)

# Torch MHA (bias=False to match PFN)
torch_mha = nn.MultiheadAttention(
    embed_dim=embed_dim,
    num_heads=num_heads,
    batch_first=True,
    bias=False,
)
# PFN wrapper
pfn_mha = PFNMultiheadAttentionV2Wrapper(
    embed_dim=embed_dim,
    num_heads=num_heads,
    batch_first=True,
    bias=False,
    config=cfg,
)

# Align PFN weights with torch weights
converted = PFNMultiHeadAttentionV2.convert_torch_nn_multihead_attention_state_dict(
    torch_mha.state_dict(),
    nhead=num_heads,
)
pfn_mha.core.load_state_dict(converted)

# Inputs (note: PFN requires key == value)
query = torch.randn(batch_size, seq_len, embed_dim)
key = value = torch.randn(batch_size, seq_len, embed_dim)

# Forward
torch_out, torch_attn = torch_mha(query, key, value, need_weights=True)
pfn_out, pfn_attn = pfn_mha(query, key, value, need_weights=True, average_attn_weights=False)

# Compare
diff = (torch_out - pfn_out).abs().max().item()
print("torch_out shape:", torch_out.shape, "pfn_out shape:", pfn_out.shape)
print("torch_attn shape:", torch_attn.shape, "pfn_attn shape:", pfn_attn.shape)
print("max |output diff|:", diff)


torch_out shape: torch.Size([2, 6, 16]) pfn_out shape: torch.Size([2, 6, 16])
torch_attn shape: torch.Size([2, 6, 6]) pfn_attn shape: torch.Size([2, 4, 6, 6])
max |output diff|: 8.940696716308594e-08


In [7]:
import torch
from torch import nn

from tfmplayground.attn_v2 import PFNAttentionConfig, MultiHeadAttention as PFNMultiHeadAttentionV2
from tfmplayground.attn_v2_wrapper import PFNMultiheadAttentionV2Wrapper

torch.manual_seed(0)

batch_size, seq_len, embed_dim, num_heads = 2, 6, 16, 4
cfg = PFNAttentionConfig(emsize=embed_dim, nhead=num_heads)

# Torch MHA with batch_first=True
torch_mha = nn.MultiheadAttention(
    embed_dim=embed_dim,
    num_heads=num_heads,
    batch_first=True,
    bias=False,
)

# PFN wrapper (also batch_first)
pfn_mha = PFNMultiheadAttentionV2Wrapper(
    embed_dim=embed_dim,
    num_heads=num_heads,
    batch_first=True,
    bias=False,
    config=cfg,
)

# Align PFN weights with torch weights
converted = PFNMultiHeadAttentionV2.convert_torch_nn_multihead_attention_state_dict(
    torch_mha.state_dict(),
    nhead=num_heads,
)
pfn_mha.core.load_state_dict(converted)

# Inputs (PFN requires key == value)
query = torch.randn(batch_size, seq_len, embed_dim)
key = value = torch.randn(batch_size, seq_len, embed_dim)

# Forward (need_weights averaged over heads to match torch default)
torch_out, torch_attn = torch_mha(query, key, value, need_weights=True)
pfn_out, pfn_attn = pfn_mha(query, key, value, need_weights=True, average_attn_weights=True)

print("torch_out shape:", torch_out.shape, "pfn_out shape:", pfn_out.shape)
print("torch_attn shape:", torch_attn.shape, "pfn_attn shape:", pfn_attn.shape)

assert torch_out.shape == pfn_out.shape, "Shape mismatch (likely batch_first issue)"
diff = (torch_out - pfn_out).abs().max().item()
print("max |output diff|:", diff)


torch_out shape: torch.Size([2, 6, 16]) pfn_out shape: torch.Size([2, 6, 16])
torch_attn shape: torch.Size([2, 6, 6]) pfn_attn shape: torch.Size([2, 6, 6])
max |output diff|: 8.940696716308594e-08


In [10]:
attn_weight_external.shape

torch.Size([2, 5, 5])

In [15]:
text_norm.shape

torch.Size([2, 5, 8])

In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
import torch
from tfmplayground.attn_v2 import PFNAttentionConfig, MultiHeadAttention

torch.manual_seed(0)
# Look back window size 3
# Config/instance
batch_size, seq_len, embed_dim, num_heads = 1, 3, 32, 4
head_dim = embed_dim // num_heads
cfg = PFNAttentionConfig(emsize=embed_dim, nhead=num_heads)
attn = MultiHeadAttention(
    d_k=head_dim,
    d_v=head_dim,
    device=None,
    dtype=None,
    config=cfg,
)

# Inputs (key == value for PFN)
x = torch.randn(batch_size, seq_len, embed_dim)

q, k, v, kv, qkv = attn.compute_qkv(
    x,
    x,
    k_cache=None,
    v_cache=None,
    kv_cache=None,
    cache_kv=False,
    use_cached_kv=False,
    reuse_first_head_kv=False,
)

# External attention from dummy text embeddings
text_feats = torch.randn(batch_size, seq_len, 8)  # pretend text embeddings
text_norm = torch.nn.functional.normalize(text_feats, dim=-1)
sim = torch.einsum("bld,bmd->blm", text_norm, text_norm)  # [B, L, L]
attn_weight_external = torch.softmax(sim, dim=-1)  # [B, Lq, Lk]
external_gate = 0.6

# # Compute attention heads with external weight
# heads_ext = attn.compute_attention_heads(
#     q=q,
#     k=k,
#     v=v,
#     kv=kv,
#     qkv=qkv,
#     dropout_p=None,
#     softmax_scale=None,
#     attn_weight_external=attn_weight_external,
#     external_gate=external_gate,
# )  # shape [B, L, H, D]

# Base (no external) for comparison
heads_base = attn.compute_attention_heads(
    q=q,
    k=k,
    v=v,
    kv=kv,
    qkv=qkv,
    dropout_p=None,
    softmax_scale=None,
    attn_weight_external=None,
    external_gate=None,
)

# print("heads_ext shape:", heads_ext.shape)
print("heads_base shape:", heads_base.shape)
# print("max |delta|:", (heads_ext - heads_base).abs().max().item())

heads_base shape: torch.Size([1, 3, 4, 8])


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# from tabdpt import TabDPTRegressor
import numpy as np

X, y = fetch_california_housing(return_X_y=True)
X, y = X[:5], y[:5]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [14]:
import torch
from tfmplayground.attn_v2 import PFNAttentionConfig, MultiHeadAttention

B, L, E, H = 1, 3, 16, 4
single_eval_pos = 2  # first 2 = train, last 1 = test
cfg = PFNAttentionConfig(emsize=E, nhead=H)
attn = MultiHeadAttention(d_k=E//H, d_v=E//H, device=None, dtype=None, config=cfg)

x = torch.randn(B, L, E)
q, k, v, kv, qkv = attn.compute_qkv(
    x, x, None, None, None,
    cache_kv=False, use_cached_kv=False, reuse_first_head_kv=False,
)

# Unpack if qkv is used
if qkv is not None:
    q, k, v = qkv.unbind(dim=-3)

# Train → train
heads_train = attn.compute_attention_heads(
    q=q[:, :single_eval_pos],
    k=k[:, :single_eval_pos],
    v=v[:, :single_eval_pos],
    kv=None,
    qkv=None,
)

# Test → train
heads_test = attn.compute_attention_heads(
    q=q[:, single_eval_pos:],
    k=k[:, :single_eval_pos],
    v=v[:, :single_eval_pos],
    kv=None,
    qkv=None,
)

print(heads_train.shape, heads_test.shape)  # (B, train_len, H, d_v), (B, test_len, H, d_v)


torch.Size([1, 2, 4, 4]) torch.Size([1, 1, 4, 4])


In [13]:
import torch
from tfmplayground.attn_v2 import PFNAttentionConfig, MultiHeadAttention

torch.manual_seed(0)

# Shapes
B, L, C, H = 1, 3, 16, 4
single_eval_pos = 2  # first 2 = train, last 1 = test
head_dim = C // H
cfg = PFNAttentionConfig(emsize=C, nhead=H)

attn = MultiHeadAttention(d_k=head_dim, d_v=head_dim, device=None, dtype=None, config=cfg)

# Inputs (train rows first, then test rows)
x = torch.randn(B, L, C)

# Get q/k/v (qkv packed)
q, k, v, kv, qkv = attn.compute_qkv(
    x, x, None, None, None,
    cache_kv=False, use_cached_kv=False, reuse_first_head_kv=False,
)
if qkv is not None:
    q, k, v = qkv.unbind(dim=-3)

# External attention from dummy text features (same row order)
text_feats = torch.randn(B, L, 8)  # pretend text embeddings
text_norm = torch.nn.functional.normalize(text_feats, dim=-1)
sim = torch.einsum("bld,bmd->blm", text_norm, text_norm)  # [B, L, L]
attn_weight_external = torch.softmax(sim, dim=-1)         # [B, Lq, Lk]
external_gate = 0.6

# Slice external weights to match train/test blocks
attn_train = attn_weight_external[:, :single_eval_pos, :single_eval_pos]        # train→train
attn_test = attn_weight_external[:, single_eval_pos:, :single_eval_pos]         # test→train

# Train → train with external attn
heads_train_ext = attn.compute_attention_heads(
    q=q[:, :single_eval_pos],
    k=k[:, :single_eval_pos],
    v=v[:, :single_eval_pos],
    kv=None,
    qkv=None,
    attn_weight_external=attn_train,
    external_gate=external_gate,
)

# Test → train with external attn
heads_test_ext = attn.compute_attention_heads(
    q=q[:, single_eval_pos:],
    k=k[:, :single_eval_pos],
    v=v[:, :single_eval_pos],
    kv=None,
    qkv=None,
    attn_weight_external=attn_test,
    external_gate=external_gate,
)

print("heads_train_ext shape:", heads_train_ext.shape)  # (B, train_len, H, head_dim)
print("heads_test_ext shape:", heads_test_ext.shape)    # (B, test_len, H, head_dim)
print("max |train heads|:", heads_train_ext.abs().max().item())
print("max |test heads|:", heads_test_ext.abs().max().item())


heads_train_ext shape: torch.Size([1, 2, 4, 4])
heads_test_ext shape: torch.Size([1, 1, 4, 4])
max |train heads|: 1.5444631576538086
max |test heads|: 1.3538498878479004


In [11]:
import torch
from tfmplayground.attn_v2 import PFNAttentionConfig, MultiHeadAttention

torch.manual_seed(0)

B, L_train, L_test, C, H = 1, 2, 1, 16, 4
L = L_train + L_test
cfg = PFNAttentionConfig(emsize=C, nhead=H)
attn = MultiHeadAttention(d_k=C//H, d_v=C//H, device=None, dtype=None, config=cfg)

# numeric inputs: train rows first, then test rows
x = torch.randn(B, L, C)

# get q/k/v (unpack qkv if present)
q, k, v, kv, qkv = attn.compute_qkv(x, x, None, None, None,
                                    cache_kv=False, use_cached_kv=False, reuse_first_head_kv=False)
if qkv is not None:
    q, k, v = qkv.unbind(dim=-3)

# logits / softmax (ps): shape [B, Lq, Lk, H]
logits = torch.einsum("b q h d, b k h d -> b q k h", q, k) / (q.shape[-1] ** 0.5)
ps = torch.softmax(logits, dim=2)  # [B, L, L, H]

# external attn from text features (same row order)
text_feats = torch.randn(B, L, 8)  # pretend text embeddings
text_norm = torch.nn.functional.normalize(text_feats, dim=-1)
sim = torch.einsum("bld,bmd->blm", text_norm, text_norm)  # [B, L, L]
attn_ext_full = torch.softmax(sim, dim=-1)  # [B, L, L]

# slice external weights: train->train and test->train, then stitch back
attn_ext = torch.zeros_like(attn_ext_full)
attn_ext[:, :L_train, :L_train] = attn_ext_full[:, :L_train, :L_train]
attn_ext[:, L_train:, :L_train] = attn_ext_full[:, L_train:, :L_train]
# broadcast to match ps shape: [B, Lq, Lk, H]
attn_ext = attn_ext.unsqueeze(-1).expand_as(ps)

gate = 0.6  # blend
ps_mixed = gate * ps + (1.0 - gate) * attn_ext

# attention outputs: [B, Lq, H, d_v]
heads_ext = torch.einsum("b q k h, b k h d -> b q h d", ps_mixed, v)

print("heads_ext shape:", heads_ext.shape)  # (1, 3, 4, 4)


heads_ext shape: torch.Size([1, 3, 4, 4])


In [12]:
attn_ext_full

tensor([[[0.7229, 0.1196, 0.1575],
         [0.0989, 0.5976, 0.3035],
         [0.1263, 0.2943, 0.5794]]])

In [15]:
import torch
from tfmplayground.attn_v2 import PFNAttentionConfig, MultiHeadAttention

torch.manual_seed(0)

# Setup: 3 tokens (rows), batch=1, embed=16, 4 heads
B, L, C, H = 1, 3, 16, 4
cfg = PFNAttentionConfig(emsize=C, nhead=H)
attn = MultiHeadAttention(d_k=C // H, d_v=C // H, device=None, dtype=None, config=cfg)

# Numeric input (self-attention over 3 tokens)
x = torch.randn(B, L, C)

# Get q/k/v (unpack if stacked in qkv)
q, k, v, kv, qkv = attn.compute_qkv(x, x, None, None, None,
                                    cache_kv=False, use_cached_kv=False, reuse_first_head_kv=False)
if qkv is not None:
    q, k, v = qkv.unbind(dim=-3)

# Base attention (no external)
heads_base = attn.compute_attention_heads(q, k, v, kv=None, qkv=None)

# External attention from dummy text features (same 3 tokens)
text_feats = torch.randn(B, L, 8)
text_norm = torch.nn.functional.normalize(text_feats, dim=-1)
sim = torch.einsum("bld,bmd->blm", text_norm, text_norm)      # [B, L, L]
attn_ext = torch.softmax(sim, dim=-1)                         # [B, L, L]
gate = 0.6

# Attention with external weights
heads_ext = attn.compute_attention_heads(
    q=q, k=k, v=v, kv=None, qkv=None,
    attn_weight_external=attn_ext,
    external_gate=gate,
)

print("heads_base shape:", heads_base.shape)  # (B, L, H, d_v)
print("heads_ext  shape:", heads_ext.shape)
print("max |delta|:", (heads_ext - heads_base).abs().max().item())

heads_base shape: torch.Size([1, 3, 4, 4])
heads_ext  shape: torch.Size([1, 3, 4, 4])
max |delta|: 0.47715771198272705


In [16]:
import torch
from tfmplayground.attn_v2 import PFNAttentionConfig
from tfmplayground.attn_v2_wrapper import PFNMultiheadAttentionV2Wrapper

torch.manual_seed(0)

# Setup: 3 tokens, batch=1, embed=16, 4 heads
B, L, C, H = 1, 3, 16, 4
cfg = PFNAttentionConfig(emsize=C, nhead=H)

mha = PFNMultiheadAttentionV2Wrapper(
    embed_dim=C,
    num_heads=H,
    batch_first=True,
    bias=False,
    config=cfg,
)

# Inputs (key must equal value for PFN)
query = torch.randn(B, L, C)
key = value = torch.randn(B, L, C)

# Base forward (no external)
out_base, attn_base = mha(query, key, value, need_weights=True, average_attn_weights=False)

# External attention from dummy text features (same 3 tokens)
text_feats = torch.randn(B, L, 8)
text_norm = torch.nn.functional.normalize(text_feats, dim=-1)
sim = torch.einsum("bld,bmd->blm", text_norm, text_norm)   # [B, L, L]
attn_ext = torch.softmax(sim, dim=-1)                      # [B, L, L]
gate = 0.6

# Forward with external attention
out_ext, attn_ext_out = mha(
    query,
    key,
    value,
    need_weights=True,
    average_attn_weights=False,  # keep per-head weights
    attn_weight_external=attn_ext,
    external_gate=gate,
)

print("out_base shape:", out_base.shape, "out_ext shape:", out_ext.shape)
print("attn_base shape:", attn_base.shape, "attn_ext shape:", attn_ext_out.shape)
print("max |delta output|:", (out_ext - out_base).abs().max().item())


out_base shape: torch.Size([1, 3, 16]) out_ext shape: torch.Size([1, 3, 16])
attn_base shape: torch.Size([1, 4, 3, 3]) attn_ext shape: torch.Size([1, 4, 3, 3])
max |delta output|: 0.19496625661849976


In [1]:
import numpy as np
import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from tfmplayground.interface import NanoTabPFNRegressor

# Data (tiny slice for a quick run)
X, y = fetch_california_housing(return_X_y=True)
X, y = X[:5], y[:5]  # tiny sample
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

# Use numeric features as stand-in text features for this test
X_text_train = X_train.copy()
X_text_test = X_test.copy()

# Model
reg = NanoTabPFNRegressor(num_mem_chunks=1)

# Fit (note the X_text argument)
reg.fit(X_train, y_train, X_text_train)

# Predict (note X_text_test)
with torch.no_grad():
    preds = reg.predict(X_test, X_text_test)

print("y_test:", y_test)
print("preds :", preds)
print("r2    :", r2_score(y_test, preds))


using GPU backend
using GPU backend
missing: [] unexpected: []
y_test: [3.585 3.422]
preds : [3.291204 3.243338]
r2    : -7.9003105527729875
